In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

vers_engleza = "The sun is shining bright today,"
vers_romana = "Soarele, dintre norii cenușii,"

input_ids = tokenizer.encode(vers_engleza, return_tensors="pt")
output_ids = model.generate(
    input_ids,
    attention_mask=input_ids.ne(0),
    max_length=50, # Token-uri (cuvinte / silabe) sa aibă maxim
    temperature=0.8, # Creativitate (mai mare ca 1 e haotic, mai mic e predictibil)
    top_k=50, # Alege doar din cele mai probabile 50 de cuvinte următoare
    top_p=0.9, # Probabilitatea cuvintelor celor mai logice din vocabular
    do_sample=True, # Nu genereaza acelasi text de fiecare data
    no_repeat_ngram_size=2, # Nu repeta aceeasi pereche de cuvinte la nesfarsit
    pad_token_id=tokenizer.eos_token_id
)

poezie = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(poezie)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

The sun is shining bright today, and I'm looking out of the window at the sunset.

I look up at my friend. "This is you!" I say, looking down. He looks at me with the same expression he does now


In [ ]:
temperature = [0.1, 0.8, 1.5]
results1 = ["The sun is shining bright today, and the moon is rising. The moon has risen, but the sun has not. The sun's risen. And the earth is falling.", "The sun is shining bright today, it's the end of the world, but it doesn't really mean that much to me. I don't want to get the feeling that I'm dead or dying.", "The sun is shining bright today, but to remember a time where the light from the Sun is the most precious of things you see is, it's beautiful. Mallory said the group plans to use the funds to bring in medical experts"]

top_k = [20, 50, 100]
results2 = ["The sun is shining bright today, and we're still in the midst of the day. We're not going to be able to see it again until the sun goes down and it stops shining. We're going in and out and the", "The sun is shining bright today, and you're sitting on the porch, staring at a beautiful red house on a farm that has been torn down by the Red River. Your hands are shaking, like your hand is shaking. Your eyes are", "The sun is shining bright today, and it's time for you to get your boots on the ground! The band of four men gathered around the horse. The sun had already begun to set, but the group of women was already moving in"]

top_p = [0.2, 0.5, 0.9]
results3 = ["The sun is shining bright today, and the moon is rising. The sky is bright, but the sun has not yet risen. The sun will rise again tomorrow. And the earth will be a little brighter tomorrow, too.", "The sun is shining bright today, but we are still in the middle of the night. The sun will rise again tomorrow. It will be a long night, and we will see the sun rise. We will never see it again.", "The sun is shining bright today, and the sky is filled with birds. It's a beautiful day, you know, it's bright, but I want to see you again, said the boy."]

result_romana = "Soarele, dintre norii cenușii, ea cœra, canto, aes, ca, ac, di, si, ita, an, and aedce; and this is"

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)

data_temperature = {
    "Valoare": temperature,
    "Text Generat": results1,
}
df_temp = pd.DataFrame(data_temperature)

data_top_k = {
    "Valoare": top_k,
    "Text Generat": results2,
}
df_top_k = pd.DataFrame(data_top_k)

data_top_p = {
    "Valoare": top_p,
    "Text Generat": results3,
}
df_top_p = pd.DataFrame(data_top_p)
display(df_temp.style
        .set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'})
        .format({"Valoare": "{:.1f}"})
        )
display(df_top_k.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
display(df_top_p.style
        .set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'})
        .format({"Valoare": "{:.1f}"})
        )

In [3]:
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

ds = load_dataset("biglam/gutenberg-poetry-corpus", split="train[:10000]")

tokenizer_trained = GPT2Tokenizer.from_pretrained("gpt2")
model_trained = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer_trained.pad_token = tokenizer_trained.eos_token

def tokenize_function(examples):
    return tokenizer_trained(examples["line"], truncation=True, max_length=128)

tokenized_datasets = ds.map(tokenize_function, batched=True, remove_columns=["line"])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_trained, mlm=False)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    save_steps=500
)

trainer = Trainer(
    model=model_trained,
    args=training_args,
    train_dataset=tokenized_datasets,
    data_collator=data_collator,
)

trainer.train()
trainer.save_model("./model")
tokenizer_trained.save_pretrained("./model")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
500,4.729603
1000,4.461243
1500,4.393908
2000,4.271985
2500,4.230268
3000,3.909945
3500,3.835954
4000,3.851656
4500,3.832635
5000,3.801995


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model\\tokenizer_config.json', './model\\tokenizer.json')

In [9]:
tokenizer_poet = GPT2Tokenizer.from_pretrained("./model")
model_poet = GPT2LMHeadModel.from_pretrained("./model")
model_poet = model_poet.to("cuda")

input_ids = tokenizer_poet.encode(vers_romana, return_tensors="pt").to("cuda")
output_ids = model_poet.generate(
    input_ids,
    attention_mask=input_ids.ne(0),
    max_length=50, # Token-uri (cuvinte / silabe) sa aibă maxim
    temperature=0.5, # Creativitate (mai mare ca 1 e haotic, mai mic e predictibil)
    top_k=20, # Alege doar din cele mai probabile 50 de cuvinte următoare
    top_p=0.9, # Probabilitatea cuvintelor celor mai logice din vocabular
    do_sample=True, # Nu genereaza acelasi text de fiecare data
    no_repeat_ngram_size=2, # Nu repeta aceeasi pereche de cuvinte la nesfarsit
    pad_token_id=tokenizer.eos_token_id
)

poezie = tokenizer_poet.decode(output_ids[0], skip_special_tokens=True)
print(poezie)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Soarele, dintre norii cenușii, the great Bird, that in his flight, to see him, stole, and haste.  Thus hee, who with all his mighte hath seen,


### c. 1. Modelul general are coerenta logica si gramaticala, dar limbajul e modern si scris in proza. Are si halucinatii si deviaza de la subiect. Modelul antrenat are vocabular poetic si incearca sa obtina structura de poezie, dar calitatea logica scade drastic

### c. 2. Acesta este scenariul ideal de functionare, deoarece limba promptului coincide cu limba corpusului. Modelul recunoaste cuvintele din prompt si continua sa genereze versuri in limba engleza

### c. 3 si 4. Modelul citeste promptul dar continua imediat in limba engleza. Trateaza cuvintele in romana ca zgomot de fond. Cum modelul a fost antrenat pe cuvinte in limba engleza si nu a vazut niciun cuvant in limba romana, el forteaza iesirea la cuvinte in limba engleza

### c. 5. Antrenam modelul pe un set de date care contine exclusiv poezii despre natura